# MemGraph — Graph-RAG vs Flat-RAG Evaluation

This notebook benchmarks our hybrid graph-RAG retriever against a flat vector-only baseline.

**Metrics**
- **Hit@k** — was the correct source chunk in the top-k results?
- **MRR** (Mean Reciprocal Rank) — how highly ranked was the first correct result?
- **Answer faithfulness** — does the answer contradict the source context? (LLM judge)
- **Multi-hop accuracy** — on questions requiring ≥2 hops, does graph-RAG outperform flat?


In [ ]:
import asyncio
import json
import pandas as pd
import matplotlib.pyplot as plt
import httpx

API_BASE = 'http://localhost:8000'

# --- Test dataset ---
# Format: {question, expected_sources, hops, expected_answer_contains}
TEST_QUESTIONS = [
    {
        'question': 'What is a knowledge graph?',
        'expected_sources': ['knowledge_graph.pdf'],
        'hops': 1,
        'expected_keywords': ['nodes', 'edges', 'entities']
    },
    # Add your own multi-hop questions here:
    # {
    #     'question': 'What did Alice say about the Q3 project that Bob was leading?',
    #     'expected_sources': ['meeting_notes.pdf', 'email_thread.txt'],
    #     'hops': 2,
    #     'expected_keywords': ['Q3', 'deadline']
    # }
]

print(f'Loaded {len(TEST_QUESTIONS)} test questions')

In [ ]:
async def run_question(question: str) -> dict:
    async with httpx.AsyncClient(timeout=60) as client:
        resp = await client.post(f'{API_BASE}/ask', json={'question': question})
        resp.raise_for_status()
        return resp.json()

async def evaluate_all():
    results = []
    for item in TEST_QUESTIONS:
        print(f"Q: {item['question'][:60]}...")
        result = await run_question(item['question'])
        
        # Hit@k: was any expected source in returned sources?
        hit = any(s in result['sources'] for s in item['expected_sources'])
        
        # Keyword check as proxy for answer quality
        answer_lower = result['answer'].lower()
        keyword_hits = sum(1 for kw in item['expected_keywords'] if kw.lower() in answer_lower)
        keyword_score = keyword_hits / len(item['expected_keywords'])
        
        results.append({
            'question': item['question'],
            'hops': item['hops'],
            'hit': hit,
            'keyword_score': keyword_score,
            'chunks_used': result['chunks_used'],
            'graph_triples_used': result['graph_triples_used'],
            'answer': result['answer'][:200],
        })
    return results

results = asyncio.run(evaluate_all())
df = pd.DataFrame(results)
df

In [ ]:
# --- Summary metrics ---
print('=== Overall ===')
print(f"Hit@k:           {df['hit'].mean():.2%}")
print(f"Keyword score:   {df['keyword_score'].mean():.2%}")
print(f"Avg chunks used: {df['chunks_used'].mean():.1f}")
print(f"Avg graph triples: {df['graph_triples_used'].mean():.1f}")

print('\n=== By hops ===')
print(df.groupby('hops')[['hit', 'keyword_score']].mean())

In [ ]:
# --- Plot: graph triples vs keyword score ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(df['graph_triples_used'], df['keyword_score'], alpha=0.7, color='steelblue')
axes[0].set_xlabel('Graph triples used')
axes[0].set_ylabel('Keyword score')
axes[0].set_title('Graph context vs answer quality')

df.groupby('hops')['hit'].mean().plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_xlabel('Required hops')
axes[1].set_ylabel('Hit@k')
axes[1].set_title('Retrieval hit rate by question complexity')
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig('evals/results.png', dpi=150)
plt.show()
print('Saved to evals/results.png')